<a href="https://colab.research.google.com/github/hojuelita/text_classification/blob/main/Ejemplo_vectorizaci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Métodos de embedding

In [51]:
# Instalaciones
!pip install gensim
!pip install faiss-cpu
!pip install sentence_transformers

# Bibliotecas usadas
from collections import Counter
import numpy as np

# recuperamos el texto desde el git
import requests
import pandas as pd
import re # Added import for regular expressions

import gensim
from gensim.models import Word2Vec
import nltk
#import spacy

url_texto = "https://raw.githubusercontent.com/hojuelita/text_classification/refs/heads/main/data/Cuentos/A_una_bogotana.txt"
response = requests.get(url_texto)
texto = response.text

print("Original text:")

def clean_text(text, minusculas=True):
    # Remove everything inside brackets [...]
    text = re.sub(r"\[.*\]", '', text)
    # Remove everything inside parenthesis (...)
    text = re.sub(r'\(.*\)', '', text)
    # Remove continuous asterisks *
    text = re.sub(r'\*+', '', text)
    # Remove puncuaction signs
    text = re.sub(r'\á','a',text)
    text = re.sub(r'\é','e',text)
    text = re.sub(r'\í','i',text)
    text = re.sub(r'\ó','o',text)
    text = re.sub(r'\ú','u',text)
    text = re.sub(r'\,','',text)
    text = re.sub(r'\:','',text)
    text = re.sub(r'\;','',text)
    text = re.sub(r'\-',' ',text)
    text = re.sub(r'\.','',text)
    text = re.sub(r'\¡','',text)
    text = re.sub(r'\!','',text)
    text = re.sub(r'\¿','',text)
    text = re.sub(r'\?','',text)
    # Optional: remove leading/trailing whitespace that might result from cleaning
    if(minusculas):
      text = text.lower()
    text = text.strip()
    return text

def split_paragraphs(text):
    """
    Split the text into paragraphs based on double newlines.
    """
    paragraphs = text.split('\n\n')
    splitted = {}
    for j,par in enumerate(paragraphs):
      splitted[j] = re.sub("\n", "", par.strip())
    return splitted

cleaned_texto = clean_text(texto, minusculas=True)
print("\nCleaned text:")
print(cleaned_texto)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 32.5 MB/s  0:00:00
Original text:

Cleaned text:
a una bogotana 

el pasillo señora hermosa niña es como un lento y rosado vals vea
usted como aquellos dos enamorados pueden llevar el compas en medio de
la mas ardiente conversacion el dice que los lindos ojos de una mujer
valen por todos los astros y los lindos labios por todas las rosas
como ella quiere demostrar lo contrario le mira con los bellisimos ojos
suyos le sonrie con sus inefables labios que son en un todo iguales a
aquellos con que la señorita de abril dio el primer beso al caballero de
mayo el pasillo señora hermosa niña es como un lento y rosado vals

oh si si la fuerza de una pasion es mayor infinitas veces que el
empuje de ese enorme y poderoso tequendama usted conoce la catarata

dicen que sus aguas saltan de un clima a otro que alla abajo hay palmas
y flores que arriba en la roca que conocio la espada de bolivar hace
frio que delicia estar alla abajo señora dos 

In [43]:
# vamos a limpiar el texto quitando caraceres especiales,
# todo lo que esté entre []
parrafos = split_paragraphs(cleaned_texto)
parrafos


{0: 'a una bogotana',
 1: 'el pasillo señora hermosa niña es como un lento y rosado vals veausted como aquellos dos enamorados pueden llevar el compas en medio dela mas ardiente conversacion el dice que los lindos ojos de una mujervalen por todos los astros y los lindos labios por todas las rosascomo ella quiere demostrar lo contrario le mira con los bellisimos ojossuyos le sonrie con sus inefables labios que son en un todo iguales aaquellos con que la señorita de abril dio el primer beso al caballero demayo el pasillo señora hermosa niña es como un lento y rosado vals',
 2: 'oh si si la fuerza de una pasion es mayor infinitas veces que elempuje de ese enorme y poderoso tequendama usted conoce la catarata',
 3: 'dicen que sus aguas saltan de un clima a otro que alla abajo hay palmasy flores que arriba en la roca que conocio la espada de bolivar hacefrio que delicia estar alla abajo señora dos que se quieren lasoberana armonia de la naturaleza pondria un palio augusto y soberbio alidili

Una vez que tengamos los párrafos ya separados podemos vectorizarlos, lo vamos a hacer primero con word2vec pero también vamos a probarlo con FastText y Roberta

## Tokenization

In [44]:
# modelo para oraciones de nltk
nltk.download('punkt')
nltk.download('punkt_tab')

# revisar el modelo usado por spacy

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [49]:
# Tokenize
oraciones = nltk.sent_tokenize(cleaned_texto, language='spanish')
oraciones_tokenizadas = [nltk.word_tokenize(sent, language='spanish') for sent in oraciones]

# Quitamos lo que no sean oraciones o palabras
oraciones_tokenizadas_limpias = [
    [re.sub(r'[^\w\s]','',token) for token in sent if len(token) > 1]
    for sent in oraciones_tokenizadas
]

print(len(f'Cantidad de oraciones {len(oraciones_tokenizadas_limpias)}'))
oraciones_tokenizadas_limpias

23


[['una',
  'bogotana',
  'el',
  'pasillo',
  'señora',
  'hermosa',
  'niña',
  'es',
  'como',
  'un',
  'lento',
  'rosado',
  'vals',
  'vea',
  'usted',
  'como',
  'aquellos',
  'dos',
  'enamorados',
  'pueden',
  'llevar',
  'el',
  'compas',
  'en',
  'medio',
  'de',
  'la',
  'mas',
  'ardiente',
  'conversacion',
  'el',
  'dice',
  'que',
  'los',
  'lindos',
  'ojos',
  'de',
  'una',
  'mujer',
  'valen',
  'por',
  'todos',
  'los',
  'astros',
  'los',
  'lindos',
  'labios',
  'por',
  'todas',
  'las',
  'rosas',
  'como',
  'ella',
  'quiere',
  'demostrar',
  'lo',
  'contrario',
  'le',
  'mira',
  'con',
  'los',
  'bellisimos',
  'ojos',
  'suyos',
  'le',
  'sonrie',
  'con',
  'sus',
  'inefables',
  'labios',
  'que',
  'son',
  'en',
  'un',
  'todo',
  'iguales',
  'aquellos',
  'con',
  'que',
  'la',
  'señorita',
  'de',
  'abril',
  'dio',
  'el',
  'primer',
  'beso',
  'al',
  'caballero',
  'de',
  'mayo',
  'el',
  'pasillo',
  'señora',
  'hermosa'

In [39]:
# tokenizacion en palabras
corpus = []
corpus_oraciones = []
for parrafo in parrafos.values():
  print(parrafo)
  try:
    corpus_oraciones.append( nltk.sent_tokenize(parrafo)[0].split() )
  except:
    print(parrafo)
  oraciones = [palabra for oracion in nltk.sent_tokenize(parrafo) for palabra in oracion.split()]
  corpus += oraciones
  print(corpus)

a una bogotana
['a', 'una', 'bogotana']
el pasillo señora hermosa niña es como un lento y rosado vals veausted como aquellos dos enamorados pueden llevar el compas en medio dela mas ardiente conversacion el dice que los lindos ojos de una mujervalen por todos los astros y los lindos labios por todas las rosascomo ella quiere demostrar lo contrario le mira con los bellisimos ojossuyos le sonrie con sus inefables labios que son en un todo iguales aaquellos con que la señorita de abril dio el primer beso al caballero demayo el pasillo señora hermosa niña es como un lento y rosado vals
['a', 'una', 'bogotana', 'el', 'pasillo', 'señora', 'hermosa', 'niña', 'es', 'como', 'un', 'lento', 'y', 'rosado', 'vals', 'veausted', 'como', 'aquellos', 'dos', 'enamorados', 'pueden', 'llevar', 'el', 'compas', 'en', 'medio', 'dela', 'mas', 'ardiente', 'conversacion', 'el', 'dice', 'que', 'los', 'lindos', 'ojos', 'de', 'una', 'mujervalen', 'por', 'todos', 'los', 'astros', 'y', 'los', 'lindos', 'labios', 'po

In [50]:
# Cantidad de tokens en este escrito
cantidad_tokens = Counter([token for oracion in oraciones_tokenizadas_limpias for token in oracion])

print(f'Cantidad de tokens {len(cantidad_tokens)}')
print(f'Populares {cantidad_tokens.most_common(10)}')

Cantidad de tokens 240
Populares [('de', 20), ('que', 20), ('la', 17), ('el', 15), ('un', 12), ('en', 10), ('una', 8), ('señora', 8), ('los', 8), ('como', 7)]


In [56]:
# Importamos las bibliotecas para transformers
import faiss
import torch
import gensim
from gensim.models import Word2Vec, FastText

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel, AutoConfig
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

## Word2Vect

In [59]:
# instanciamos el modelo
# usando las oraciones
word2vec_model = Word2Vec(
    sentences=oraciones_tokenizadas_limpias,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg = 1,
    epochs = 10
    )


In [60]:

print(f"Vocabulario Word2Vec: {len(word2vec_model.wv)}")

# palabras similares
test_words = ['hermosa','mujer','genuflexión']

for word in test_words:
    if word in word2vec_model.wv:
        print(f"\nSimilar a '{word}':")
        similar = word2vec_model.wv.most_similar(word, topn=5)
        for sim_word, score in similar:
            print(f"  {sim_word}: {score:.3f}")
    else:
        print(f"'{word}' no en el vocab")

Vocabulario Word2Vec: 54

Similar a 'hermosa':
  la: 0.579
  se: 0.541
  que: 0.533
  una: 0.522
  un: 0.518

Similar a 'mujer':
  un: 0.576
  que: 0.564
  una: 0.558
  la: 0.539
  lento: 0.537
'genuflexión' no en el vocab


In [63]:
def get_sentence_embedding_word2vec(sentence_tokens, model):
    vectors = []
    for token in sentence_tokens:
        if token in model.wv:
            vectors.append(model.wv[token])
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.vector_size)

oracion = oraciones_tokenizadas_limpias[0]
vector_oracion = get_sentence_embedding_word2vec(oracion, word2vec_model)
print(f"\nOracion ejemplo: {' '.join(oracion[:10])}...")
print(f"Word2Vec vector: {vector_oracion.shape}")


Oracion ejemplo: una bogotana el pasillo señora hermosa niña es como un...
Word2Vec vector: (100,)


In [64]:
print(vector_oracion)

[ 5.5377622e-04  5.0277077e-03 -1.2175739e-03  1.3308097e-03
  5.6840093e-03 -6.3665542e-03  4.0594670e-03  1.4674760e-02
 -9.7551160e-03 -8.4878337e-03  3.7217289e-03 -1.0845464e-02
  1.6778058e-03  7.0620542e-03  4.1443347e-03 -3.9337347e-03
  1.2281413e-02  1.2335400e-03 -1.0783214e-02 -1.5717370e-02
  3.3572083e-03  2.2969772e-03  1.4517613e-02 -1.0886304e-02
  2.3482444e-03 -2.8144763e-04 -9.1618290e-03 -2.3373531e-03
 -2.7082050e-03 -6.4016035e-04  8.6970543e-03 -4.8660273e-03
  6.2509039e-03 -1.7127292e-02 -1.2732259e-03  7.3316004e-03
 -5.6129170e-04  2.2016440e-03 -3.6934894e-03 -9.2924247e-03
 -1.9469418e-04 -7.2164652e-03 -5.6639547e-05  4.9941395e-03
  5.7229935e-03 -3.6807607e-03 -6.6544428e-03 -8.4872926e-03
  1.7213644e-03  6.0028466e-03  4.8765806e-03 -9.2289150e-03
 -5.8979886e-03 -2.6746688e-03 -1.5489890e-03 -3.1795404e-03
  8.6758109e-03 -4.2113438e-04 -1.6644063e-03  1.2877248e-03
  1.3893604e-03 -8.0340225e-03  1.1150511e-02  6.8627670e-03
 -9.4714537e-03  1.68448

## FastText

In [65]:
# Train FastText model
fasttext_model = FastText(
    sentences=oraciones_tokenizadas_limpias,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,                  # Use skip-gram
    epochs=10,
    min_n=3,
    max_n=6
)

print(f"FastText : {len(fasttext_model.wv)}")

FastText : 54


In [66]:
# W2V comparación
for word in test_words:
    print(f"\nPalabra '{word}':")

    # W2V
    if word in word2vec_model.wv:
        w2v_similar = word2vec_model.wv.most_similar(word, topn=3)
        print(f"  Word2Vec: {[w[0] for w in w2v_similar]}")

    # FastText
    if word in fasttext_model.wv:
        ft_similar = fasttext_model.wv.most_similar(word, topn=3)
        print(f"  FastText: {[w[0] for w in ft_similar]}")

    oov_word = word + "mente"  # OOV out of vocab
    print(f"\n  Palabra derivada '{oov_word}':")
    try:
        ft_oov_similar = fasttext_model.wv.most_similar(oov_word, topn=3)
        print(f"    Similaridad: {[(w[0], f'{w[1]:.3f}') for w in ft_oov_similar]}")
    except:
        print(f"    '{oov_word}' not found even with FastText")

# Embedding
def get_sentence_embedding_fasttext(sentence_tokens, model):
    vectors = []
    for token in sentence_tokens:
        vectors.append(model.wv[token])
    return np.mean(vectors, axis=0)

oracion_embedding = get_sentence_embedding_fasttext(oracion, fasttext_model)
print(f"\nFastText embedding shape: {oracion_embedding.shape}")


Palabra 'hermosa':
  Word2Vec: ['la', 'se', 'que']
  FastText: ['la', 'rosas', 'un']

  Palabra derivada 'hermosamente':
    Similaridad: [('hermosa', '0.920'), ('del', '0.858'), ('la', '0.852')]

Palabra 'mujer':
  Word2Vec: ['un', 'que', 'una']
  FastText: ['una', 'rosas', 'la']

  Palabra derivada 'mujermente':
    Similaridad: [('mujer', '0.806'), ('una', '0.805'), ('lento', '0.805')]

Palabra 'genuflexión':
  FastText: ['le', 'heridas', 'del']

  Palabra derivada 'genuflexiónmente':
    Similaridad: [('aquellos', '0.210'), ('del', '0.190'), ('es', '0.187')]

FastText embedding shape: (100,)


##RoBERTa